# Hybrid Newton-Net + Amortized Variational Inference

End-to-end pipeline for training a Transformer-based inference network on
GMM tau-p reflectivity data, then using the network's posterior as a warm
start for full-Newton Levenberg-Marquardt inversion.

**Steps:**
1. Load YAML configuration
2. Generate training data (perturbed earth models + GMM forward)
3. Train the SeismicInferenceNet (AdamW + cosine annealing + early stopping)
4. Evaluate on held-out test set (NLL, MAE, calibration)
5. Run hybrid inversion (network warm start + Newton refinement)
6. Compare convergence against random-start inversion

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# Ensure the project root is on sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Load Configuration

In [ ]:
from NeuralInversion.inference_config import load_inference_config

cfg = load_inference_config(Path("configs/default_inference.yaml"))

# Architecture summary
a = cfg.architecture
print(
    f"Architecture: d_model={a.d_model}, n_heads={a.n_heads}, "
    f"n_layers={a.n_layers}, d_ff={a.d_ff}, dropout={a.dropout}"
)

# Training summary
t = cfg.training
print(
    f"Training:     epochs={t.n_epochs}, batch_size={t.batch_size}, "
    f"lr={t.learning_rate:.0e}, patience={t.patience}"
)

# Data summary
d = cfg.data
print(f"Data:         train={d.n_train}, val={d.n_val}, test={d.n_test}")
print(f"              p_values={len(d.p_values)} slownesses, nfreq={d.nfreq}")
print(
    f"              perturbations: vel={d.velocity_perturbation:.0%}, "
    f"rho={d.density_perturbation:.0%}, h={d.thickness_perturbation:.0%}"
)

## Generate Training Data

In [ ]:
import time

import torch

from Kennett_Reflectivity.kennett_seismogram import default_ocean_crust_model

from NeuralInversion.inference_data import generate_training_data

ref_model = default_ocean_crust_model()

t0 = time.perf_counter()
paths = generate_training_data(cfg.data_dir, cfg.data, ref_model=ref_model)
datagen_time = time.perf_counter() - t0

for split, path in paths.items():
    data = torch.load(path, weights_only=True)
    n = data["log_params"].shape[0]
    r_shape = data["reflectivity"].shape
    print(
        f"{split:>5}: {n:,} samples, "
        f"reflectivity {r_shape[1:]} (n_p x 2*nfreq), "
        f"params {data['log_params'].shape[1]}"
    )

print(f"\nData generation: {datagen_time:.1f} s")

train_path = paths["train"]
val_path = paths["val"]
test_path = paths["test"]

## Train Network

In [ ]:
from NeuralInversion.inference_train import train_inference_net

t0 = time.perf_counter()
model, history = train_inference_net(cfg, train_path, val_path)
training_time = time.perf_counter() - t0

best_epoch = int(np.argmin(history["val_loss"])) + 1
best_val = min(history["val_loss"])
print(f"\nBest epoch: {best_epoch}")
print(f"Best val loss: {best_val:.4f}")
print(f"Total epochs trained: {len(history['train_loss'])}")
print(f"Training time: {training_time:.1f} s")

## Training Curves

In [ ]:
from Kennett_Reflectivity.taup_inversion import _LATEX_RCPARAMS

epochs = np.arange(1, len(history["train_loss"]) + 1)

with plt.rc_context(_LATEX_RCPARAMS):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Loss curves
    ax1.plot(epochs, history["train_loss"], "b-", label="Train", linewidth=1.0)
    ax1.plot(epochs, history["val_loss"], "r-", label="Validation", linewidth=1.0)
    ax1.axvline(
        best_epoch,
        color="gray",
        linestyle="--",
        alpha=0.5,
        label=f"Best epoch {best_epoch}",
    )
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Gaussian NLL")
    ax1.set_title("Training / Validation Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Learning rate schedule
    ax2.plot(epochs, history["learning_rate"], "k-", linewidth=1.0)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Learning Rate")
    ax2.set_title("Cosine Annealing Schedule")
    ax2.ticklabel_format(axis="y", style="sci", scilimits=(-3, -3))
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## Evaluate on Test Set

In [ ]:
from NeuralInversion.inference_train import evaluate_inference_net

metrics = evaluate_inference_net(model, test_path)

print(f"Test NLL:              {metrics['nll']:.4f}")
print(f"Mean absolute error:   {metrics['mean_abs_error']:.4f}")
print(f"Calibration (2-sigma): {metrics['calibration_2sigma']:.1%}")

## Hybrid Inversion

In [ ]:
from NeuralInversion.hybrid_inversion import hybrid_invert_taup

true_model = default_ocean_crust_model()

t0 = time.perf_counter()
hybrid_result = hybrid_invert_taup(
    true_model=true_model,
    model=model,
    p_values=cfg.data.p_values,
    nfreq=cfg.data.nfreq,
    max_iter=cfg.hybrid.max_newton_iter,
    tol=cfg.hybrid.newton_tol,
    initial_damping=cfg.hybrid.initial_damping,
    use_laplace=cfg.hybrid.use_laplace,
)
hybrid_time = time.perf_counter() - t0

inv = hybrid_result.inversion_result
print(f"\nNetwork param error: {hybrid_result.network_param_error:.4e}")
print(f"Converged: {inv.converged}")
print(f"Newton iterations: {hybrid_result.newton_iterations}")
print(f"Final misfit: {inv.misfit_history[-1]:.2e}")
print(f"Final param error: {inv.param_error_history[-1]:.4e}")
print(f"Hybrid inversion time: {hybrid_time:.2f} s")

## Comparison: Hybrid vs Random Start

In [ ]:
from GlobalMatrix.taup_inversion import invert_taup

t0 = time.perf_counter()
random_result = invert_taup(
    true_model=true_model,
    p_values=cfg.data.p_values,
    nfreq=cfg.data.nfreq,
    perturbation=cfg.data.velocity_perturbation,
    max_iter=50,
    seed=cfg.data.seed,
)
random_time = time.perf_counter() - t0

print(f"\n{'':>20} {'Hybrid':>10} {'Random':>10}")
print("-" * 42)
print(
    f"{'Iterations':>20} {hybrid_result.newton_iterations:>10} "
    f"{random_result.n_iterations:>10}"
)
print(f"{'Converged':>20} {str(inv.converged):>10} {str(random_result.converged):>10}")
print(
    f"{'Final misfit':>20} {inv.misfit_history[-1]:>10.2e} "
    f"{random_result.misfit_history[-1]:>10.2e}"
)
print(
    f"{'Final param error':>20} {inv.param_error_history[-1]:>10.4e} "
    f"{random_result.param_error_history[-1]:>10.4e}"
)
print(f"{'Wall time (s)':>20} {hybrid_time:>10.2f} {random_time:>10.2f}")

## Convergence Comparison

## Amortized Cost Analysis

The hybrid approach has a large upfront cost (data generation + training) that
must be amortized over many inversions to break even against the random-start
Newton solver. Below we compute the break-even point.

In [ ]:
upfront_cost = datagen_time + training_time
time_saved_per_inversion = random_time - hybrid_time

print("Wall-clock budget")
print("=" * 50)
print(f"  Data generation:        {datagen_time:>8.1f} s")
print(f"  Network training:       {training_time:>8.1f} s")
print(f"  Total upfront cost:     {upfront_cost:>8.1f} s")
print()
print(f"  Single hybrid inversion:  {hybrid_time:>6.2f} s")
print(f"  Single random inversion:  {random_time:>6.2f} s")
print(f"  Time saved per inversion: {time_saved_per_inversion:>6.2f} s")
print()

if time_saved_per_inversion > 0:
    breakeven = int(np.ceil(upfront_cost / time_saved_per_inversion))
    print(f"  Break-even after {breakeven:,} inversions")
else:
    breakeven = None
    print("  Hybrid is slower per-inversion — no break-even point")

print()
print("The hybrid approach is justified when:")
print("  1. Many inversions share the same model class (amortized inference)")
print("  2. The network provides calibrated uncertainty (free posterior)")
print("  3. Multi-modal posteriors risk trapping random-start Newton")

In [ ]:
with plt.rc_context(_LATEX_RCPARAMS):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    h_iters = np.arange(1, len(inv.misfit_history) + 1)
    r_iters = np.arange(1, len(random_result.misfit_history) + 1)

    # Misfit
    axes[0].semilogy(h_iters, inv.misfit_history, "b-o", markersize=4, label="Hybrid")
    axes[0].semilogy(
        r_iters,
        random_result.misfit_history,
        "r-s",
        markersize=3,
        alpha=0.7,
        label="Random start",
    )
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel(r"Misfit $\chi$")
    axes[0].set_title("Misfit convergence")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Gradient norm
    axes[1].semilogy(
        h_iters, inv.grad_norm_history, "b-o", markersize=4, label="Hybrid"
    )
    axes[1].semilogy(
        r_iters,
        random_result.grad_norm_history,
        "r-s",
        markersize=3,
        alpha=0.7,
        label="Random start",
    )
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel(r"$\|\nabla\chi\|$")
    axes[1].set_title("Gradient norm")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Parameter error
    axes[2].semilogy(
        h_iters, inv.param_error_history, "b-o", markersize=4, label="Hybrid"
    )
    axes[2].semilogy(
        r_iters,
        random_result.param_error_history,
        "r-s",
        markersize=3,
        alpha=0.7,
        label="Random start",
    )
    axes[2].set_xlabel("Iteration")
    axes[2].set_ylabel(
        r"$\|\mathbf{m} - \mathbf{m}_\mathrm{true}\|"
        r" / \|\mathbf{m}_\mathrm{true}\|$"
    )
    axes[2].set_title("Relative parameter error")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## Uncertainty Estimates

In [ ]:
# Parameter names: Vp(1-4), Vs(1-4), rho(1-4), h(1-3)
layer_names = ["Sed", "Crust", "Mantle", "HS"]
param_names = (
    [f"$V_P^\\mathrm{{{n}}}$" for n in layer_names]
    + [f"$V_S^\\mathrm{{{n}}}$" for n in layer_names]
    + [f"$\\rho^\\mathrm{{{n}}}$" for n in layer_names]
    + [f"$h^\\mathrm{{{n}}}$" for n in layer_names[:3]]
)

sigma = hybrid_result.network_sigma
x_pos = np.arange(len(param_names))

with plt.rc_context(_LATEX_RCPARAMS):
    fig, ax = plt.subplots(figsize=(12, 5))

    bars = ax.bar(
        x_pos, sigma, color="#2980b9", edgecolor="black", linewidth=0.5, alpha=0.8
    )
    ax.set_xticks(x_pos)
    ax.set_xticklabels(param_names, rotation=45, ha="right")
    ax.set_ylabel(r"Network $\sigma$ (log-space)")
    ax.set_title("Amortized Posterior Uncertainty per Parameter")
    ax.grid(True, alpha=0.3, axis="y")

    # Group separators
    for boundary in [3.5, 7.5, 11.5]:
        ax.axvline(boundary, color="gray", linestyle=":", alpha=0.5)

    fig.tight_layout()
    plt.show()

## Export Results

In [ ]:
from Kennett_Reflectivity.taup_inversion import plot_convergence_curves

from NeuralInversion.inference_config import save_inference_config

outdir = Path("figures")
outdir.mkdir(parents=True, exist_ok=True)

# Convergence PDF (hybrid result)
plot_convergence_curves(inv, outdir / "hybrid_convergence.pdf")
print(f"Convergence -> {outdir / 'hybrid_convergence.pdf'}")

# Checkpoint info
ckpt_path = cfg.checkpoint_dir / "best_model.pt"
print(f"Best checkpoint: {ckpt_path.resolve()}")
print(f"  epoch={best_epoch}, val_loss={best_val:.4f}")

# Config YAML for reproducibility
save_inference_config(cfg, outdir / "inference_config.yaml")
print(f"Config saved -> {outdir / 'inference_config.yaml'}")